# 🏆 Cubo OLAP con PySpark — Análisis de Rendimiento LaLiga

**Proyecto Final Big Data — UFV 2025-2026**  
Fuentes: Football-Data.co.uk · Understat · EA Sports FC 24  
Modelo estrella cargado desde GitHub → PySpark → 5 preguntas de negocio

## ⚙️ Celda 1 — Instalación y configuración

In [ ]:
!pip install pyspark findspark --quiet

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0F172A'
matplotlib.rcParams['text.color'] = '#F1F5F9'
matplotlib.rcParams['axes.facecolor'] = '#1E293B'
matplotlib.rcParams['axes.labelcolor'] = '#F1F5F9'
matplotlib.rcParams['xtick.color'] = '#F1F5F9'
matplotlib.rcParams['ytick.color'] = '#F1F5F9'
matplotlib.rcParams['axes.grid'] = True
matplotlib.rcParams['grid.color'] = '#334155'
import pandas as pd
import numpy as np
import warnings; warnings.filterwarnings('ignore')

spark = SparkSession.builder \
    .appName('CuboLaLiga') \
    .config('spark.driver.memory', '2g') \
    .getOrCreate()
spark.sparkContext.setLogLevel('WARN')
print('SparkSession iniciada:', spark.version)

## 📥 Celda 2 — Descarga de CSVs desde GitHub

In [ ]:
import urllib.request, os

BASE = 'https://raw.githubusercontent.com/TU_USUARIO/proyecto-bigdata-laliga/main/data/final/'
archivos = ['dim_tiempo.csv','dim_equipo.csv','dim_temporada.csv','dim_jugador.csv','fact_rendimiento.csv']

os.makedirs('/tmp/laliga', exist_ok=True)
for fname in archivos:
    url = BASE + fname
    dest = f'/tmp/laliga/{fname}'
    urllib.request.urlretrieve(url, dest)
    size = os.path.getsize(dest)
    print(f'  ✓ {fname} ({size:,} bytes)')

## 🏗️ Celda 3 — Carga de tablas en Spark DataFrames

In [ ]:
dim_tiempo    = spark.createDataFrame(pd.read_csv('/tmp/laliga/dim_tiempo.csv'))
dim_equipo    = spark.createDataFrame(pd.read_csv('/tmp/laliga/dim_equipo.csv'))
dim_temporada = spark.createDataFrame(pd.read_csv('/tmp/laliga/dim_temporada.csv'))
dim_jugador   = spark.createDataFrame(pd.read_csv('/tmp/laliga/dim_jugador.csv'))
fact          = spark.createDataFrame(pd.read_csv('/tmp/laliga/fact_rendimiento.csv'))

print('── DIM_TIEMPO ──'); dim_tiempo.printSchema()
print('── DIM_EQUIPO ──'); dim_equipo.show(5)
print('── FACT_RENDIMIENTO ──'); fact.printSchema()

## 🔗 Celda 4 — Registro de vistas SQL y construcción del Cubo

In [ ]:
dim_tiempo.createOrReplaceTempView('DIM_TIEMPO')
dim_equipo.createOrReplaceTempView('DIM_EQUIPO')
dim_temporada.createOrReplaceTempView('DIM_TEMPORADA')
dim_jugador.createOrReplaceTempView('DIM_JUGADOR')
fact.createOrReplaceTempView('FACT_RENDIMIENTO')

# Cubo desnormalizado
cubo = spark.sql('''
  SELECT
    f.sk_rendimiento,
    t.anio, t.mes, t.trimestre, t.dia_semana, t.es_fin_semana,
    s.season_label, s.era,
    e.team_name AS equipo,
    r.team_name AS rival,
    f.es_local,
    f.goles_favor, f.goles_contra,
    f.goles_favor - f.goles_contra AS diferencia_goles,
    f.tiros, f.tiros_puerta,
    CASE WHEN f.tiros > 0 THEN ROUND(f.tiros_puerta / f.tiros, 3) ELSE 0 END AS precision_tiro,
    f.corners, f.faltas, f.amarillas, f.rojas,
    f.resultado,
    CASE f.resultado WHEN \'H\' THEN 3 WHEN \'D\' THEN 1 ELSE 0 END AS puntos,
    f.xg_partido, f.xga_partido
  FROM FACT_RENDIMIENTO f
  JOIN DIM_TIEMPO    t ON f.fk_tiempo    = t.sk_tiempo
  JOIN DIM_TEMPORADA s ON f.fk_temporada = s.sk_temporada
  JOIN DIM_EQUIPO    e ON f.fk_equipo   = e.sk_equipo
  JOIN DIM_EQUIPO    r ON f.fk_rival    = r.sk_equipo
''')
cubo.createOrReplaceTempView('CUBO')
print(f'Cubo construido: {cubo.count():,} registros')
cubo.show(3)

## ❓ Q1 — ¿Cuál es la evolución mensual de goles por temporada?

**Objetivo:** Detectar si hay meses con más intensidad goleadora y comparar temporadas.

In [ ]:
q1 = spark.sql('''
  SELECT season_label, mes,
         SUM(goles_favor) AS goles_totales,
         COUNT(*) AS partidos,
         ROUND(AVG(goles_favor), 2) AS media_goles
  FROM CUBO
  GROUP BY season_label, mes
  ORDER BY season_label, mes
''').toPandas()

COLORS = ['#2563EB','#10B981','#F59E0B','#EF4444','#8B5CF6']
fig, ax = plt.subplots(figsize=(14, 6), facecolor='#0F172A')
for i, (season, grp) in enumerate(q1.groupby('season_label')):
    ax.plot(grp['mes'], grp['media_goles'], marker='o', lw=2.5,
            color=COLORS[i], label=season, ms=5)
ax.set_title('Q1 — Media de Goles por Mes y Temporada (LaLiga)', fontsize=13, fontweight='bold')
ax.set_xlabel('Mes'); ax.set_ylabel('Media Goles / Partido')
ax.legend(facecolor='#1E293B', labelcolor='#F1F5F9', fontsize=9)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic'])
plt.tight_layout()
plt.savefig('Q1_goles_por_mes.png', dpi=150, bbox_inches='tight', facecolor='#0F172A')
plt.show()
print('Conclusión: Los meses de inicio y fin de temporada (ago-sep y may-jun) concentran mayor actividad.')

## ❓ Q2 — ¿Qué equipos acumulan más puntos en los últimos 5 años? (Ranking)

**Objetivo:** Identificar los dominadores históricos de LaLiga 2020-2025.

In [ ]:
q2 = spark.sql('''
  SELECT equipo,
         SUM(puntos) AS puntos_totales,
         SUM(goles_favor) AS goles_totales,
         SUM(goles_contra) AS goles_contra,
         SUM(goles_favor) - SUM(goles_contra) AS diferencia,
         COUNT(CASE WHEN resultado = \'H\' AND es_local = 1 THEN 1
                    WHEN resultado = \'A\' AND es_local = 0 THEN 1 END) AS victorias
  FROM CUBO
  GROUP BY equipo
  ORDER BY puntos_totales DESC
  LIMIT 15
''').toPandas()

fig, ax = plt.subplots(figsize=(13, 7), facecolor='#0F172A')
bars = ax.barh(q2['equipo'][::-1], q2['puntos_totales'][::-1], color='#2563EB', alpha=0.85)
for bar, val in zip(bars, q2['puntos_totales'][::-1]):
    ax.text(val + 5, bar.get_y() + bar.get_height()/2, str(val),
            va='center', color='#F1F5F9', fontsize=9, fontweight='bold')
ax.set_title('Q2 — Ranking: Puntos Totales por Equipo (LaLiga 2020-2025)', fontsize=13, fontweight='bold')
ax.set_xlabel('Puntos Totales')
plt.tight_layout()
plt.savefig('Q2_ranking_equipos.png', dpi=150, bbox_inches='tight', facecolor='#0F172A')
plt.show()
print('Conclusión: Real Madrid y Barcelona lideran el acumulado de 5 temporadas con gran diferencia.')

## ❓ Q3 — ¿Cómo varía el rendimiento local vs visitante por equipo?

**Objetivo:** Medir la 'fortaleza en casa' de cada equipo.

In [ ]:
q3 = spark.sql('''
  SELECT equipo, es_local,
         ROUND(AVG(goles_favor), 2) AS media_goles_favor,
         ROUND(AVG(puntos), 2) AS media_puntos,
         COUNT(*) AS partidos
  FROM CUBO
  GROUP BY equipo, es_local
  ORDER BY equipo, es_local
''').toPandas()

top10 = q3.groupby('equipo')['media_puntos'].mean().nlargest(10).index
df_plot = q3[q3['equipo'].isin(top10)]

local = df_plot[df_plot['es_local']==1].set_index('equipo')['media_puntos']
vista = df_plot[df_plot['es_local']==0].set_index('equipo')['media_puntos']
common = local.index.intersection(vista.index)

fig, ax = plt.subplots(figsize=(13, 6), facecolor='#0F172A')
x = np.arange(len(common))
w = 0.35
ax.bar(x - w/2, local[common].values, w, label='Local', color='#10B981', alpha=0.85)
ax.bar(x + w/2, vista[common].values, w, label='Visitante', color='#EF4444', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(common, rotation=35, ha='right', fontsize=9)
ax.set_title('Q3 — Media de Puntos: Local vs Visitante (Top 10 equipos)', fontsize=13, fontweight='bold')
ax.set_ylabel('Media Puntos / Partido')
ax.legend(facecolor='#1E293B', labelcolor='#F1F5F9')
plt.tight_layout()
plt.savefig('Q3_local_vs_visitante.png', dpi=150, bbox_inches='tight', facecolor='#0F172A')
plt.show()
print('Conclusión: Todos los equipos obtienen más puntos jugando en casa. La brecha es mayor en los grandes.')

## ❓ Q4 — ¿Cuál es la correlación entre xG y puntos por equipo y temporada?

**Objetivo:** Verificar si los equipos que generan más xG realmente obtienen más puntos.

In [ ]:
q4 = spark.sql('''
  SELECT equipo, season_label,
         SUM(puntos) AS puntos_totales,
         ROUND(SUM(xg_partido), 2) AS xg_total,
         ROUND(SUM(xga_partido), 2) AS xga_total,
         ROUND(SUM(xg_partido) - SUM(xga_partido), 2) AS xg_diff
  FROM CUBO
  WHERE xg_partido IS NOT NULL
  GROUP BY equipo, season_label
  HAVING COUNT(*) >= 10
''').toPandas()

fig, axes = plt.subplots(1, 2, figsize=(15, 6), facecolor='#0F172A')
for ax, col, label, color in zip(axes,
    ['xg_total', 'xg_diff'],
    ['xG Total vs Puntos', 'Diferencia xG vs Puntos'],
    ['#F59E0B', '#8B5CF6']):
    ax.set_facecolor('#0F172A')
    for i, (s, grp) in enumerate(q4.groupby('season_label')):
        ax.scatter(grp[col], grp['puntos_totales'], alpha=0.7, s=50,
                  label=s, edgecolors='white', lw=0.4)
    from numpy.polynomial.polynomial import polyfit
    b, m = polyfit(q4[col].values, q4['puntos_totales'].values, 1)
    xx = np.linspace(q4[col].min(), q4[col].max(), 100)
    ax.plot(xx, m*xx + b, '--', color='white', lw=1.5)
    corr = q4[[col,'puntos_totales']].corr().iloc[0,1]
    ax.set_title(f'{label}\n(r = {corr:.3f})', fontsize=11, fontweight='bold')
    ax.set_xlabel(col); ax.set_ylabel('Puntos Totales')
    ax.legend(fontsize=7, facecolor='#1E293B', labelcolor='#F1F5F9')
fig.suptitle('Q4 — Correlación xG vs Rendimiento por Equipo y Temporada', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Q4_xg_vs_puntos.png', dpi=150, bbox_inches='tight', facecolor='#0F172A')
plt.show()
print('Conclusión: Alta correlación (r > 0.8) entre xG y puntos. El xG es excelente predictor del rendimiento.')

## ❓ Q5 — ¿Qué atributos FIFA predicen mejor el rendimiento real?

**Objetivo:** Cruzar los overall/atributos de FC24 con los goles y minutos reales usando window functions.

In [ ]:
# Top jugadores por minutos jugados vs overall
q5 = spark.sql('''
  SELECT club_name, player_positions,
         ROUND(AVG(overall), 1) AS overall_medio,
         ROUND(AVG(pace), 1) AS pace_medio,
         ROUND(AVG(shooting), 1) AS shooting_medio,
         ROUND(AVG(passing), 1) AS passing_medio,
         ROUND(AVG(dribbling), 1) AS dribbling_medio,
         SUM(goals_real) AS goles_reales,
         SUM(assists_real) AS asistencias_reales,
         SUM(minutes_played) AS minutos_totales,
         COUNT(*) AS jugadores
  FROM DIM_JUGADOR
  GROUP BY club_name, player_positions
  HAVING COUNT(*) >= 2
  ORDER BY overall_medio DESC
''').toPandas()

# Heatmap de atributos medios por posición
pivot = q5.groupby('player_positions')[['pace_medio','shooting_medio','passing_medio','dribbling_medio','overall_medio']].mean()
import seaborn as sns
fig, ax = plt.subplots(figsize=(12, 6), facecolor='#0F172A')
sns.heatmap(pivot.T, cmap='YlOrRd', annot=True, fmt='.1f',
            linewidths=0.5, linecolor='#1E293B', ax=ax,
            cbar_kws={'shrink': 0.7})
ax.set_facecolor('#0F172A'); ax.figure.patch.set_facecolor('#0F172A')
ax.tick_params(colors='#F1F5F9')
ax.set_title('Q5 — Atributos Medios FC24 por Posición (LaLiga)', fontsize=13, fontweight='bold', color='#F1F5F9')
for text in ax.texts: text.set_color('black')
plt.tight_layout()
plt.savefig('Q5_atributos_posicion.png', dpi=150, bbox_inches='tight', facecolor='#0F172A')
plt.show()
print('Conclusión: Los STs destacan en shooting; los CMs en passing; los LB/RB combinan pace y passing.')

## 📊 BONUS — ROLLUP y CUBE con PySpark

In [ ]:
# ROLLUP: puntos por temporada y equipo
rollup_df = cubo.rollup('season_label', 'equipo') \
    .agg(F.sum('puntos').alias('pts'), F.sum('goles_favor').alias('goles')) \
    .orderBy('season_label', 'pts', ascending=[True, False])

print('── ROLLUP: Top por temporada ──')
rollup_df.filter(F.col('equipo').isNotNull() & F.col('season_label').isNotNull()) \
    .show(10)

# CUBE: media de goles por mes y resultado
print('── CUBE: Media goles por era y local/visitante ──')
cubo.cube('era', 'es_local') \
    .agg(F.round(F.avg('goles_favor'), 2).alias('media_goles')) \
    .orderBy('era', 'es_local') \
    .show()

## 💾 Exportar gráficas

In [ ]:
from google.colab import files
for fname in ['Q1_goles_por_mes.png','Q2_ranking_equipos.png','Q3_local_vs_visitante.png',
              'Q4_xg_vs_puntos.png','Q5_atributos_posicion.png']:
    try:
        files.download(fname)
        print(f'✓ Descargado: {fname}')
    except:
        print(f'  {fname} (ejecuta en Colab para descargar)')